# 02_pipeline — config-driven orchestration template

Build, check, publish, and record evidence for governed Fabric data pipelines.

This notebook is intentionally thin and beginner friendly. For a normal one-source/one-target pipeline, edit the clearly marked **USER EDIT SECTION** blocks: source setup configures input tables, the transform section creates target DataFrames, and target setup configures output tables and write behavior. To add another source or target table, add another dictionary to `SOURCE_TABLES` or `TARGET_TABLES`; do not copy profiling, schema, stability, DQ, catalogue-evidence, write, lineage, or runtime-summary orchestration code.

FabricOps then enriches those source and target entries with guardrail defaults, write defaults, and DataFrames before running profiling, schema validation, stability enforcement, DQ enforcement, catalogue evidence, writes, lineage, and runtime summary from the config lists.

Flow:

1. Run `00_env_config`.
2. Import required functions.
3. Select the data agreement and capture run context.
4. Configure input tables in `SOURCE_TABLES`.
5. Review source guardrail defaults only when your project needs a different source governance policy.
6. Let source framework preparation load source DataFrames and enrich configs.
7. Run source guardrails before transformation.
8. Create target DataFrames in the transform section.
9. Configure output tables and write behavior in `TARGET_TABLES`.
10. Review target guardrail and write defaults only when your project needs a different target governance or write policy.
11. Let target framework preparation add audit columns and enrich configs.
12. Run target guardrails before writes.
13. Write targets only after all target guardrails pass.
14. Capture lineage and runtime summary evidence.

Schema, stability, and DQ remain separate guardrail concepts. The reusable orchestration helper only removes repeated notebook code.

## 1. Run `00_env_config`

Load the shared FabricOps environment, path configuration, sample metadata, and metadata lakehouse routing.

In [ ]:
%run 00_env_config


## 2. Import required functions

The notebook imports existing FabricOps callables for reads, guardrail orchestration, writes, lineage, and runtime-summary evidence.

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    get_selected_agreement,
    guardrail_summary,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_table,
    run_table_guardrails,
    stop_if_any_guardrail_failed,
    stop_if_failed,
    widget_select_agreement,
    write_lakehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_warehouse_table,
)

## 3. Select data agreement and capture run context

Select the agreement that this pipeline satisfies. The selector registers this notebook in `METADATA_NOTEBOOK_REGISTRY` using the metadata target configured by `00_env_config`. The run context values are reused by guardrail evidence, lineage, and runtime summary writes.

In [ ]:
PIPELINE_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = "CHANGE_ME_pipeline"

widget_select_agreement(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


## 4. USER EDIT SECTION — source table configuration

Most users only edit this section for sources. Update each source `table_name`, `watermark_column` when applicable, and `expected_schema`. FabricOps derives the governance `dataset_name` from `table_name`, uses `layer` as the default governance `stage`, and uses `None` as the default `watermark_value`.

Valid FabricOps layer/stage concepts:

- `source` = raw/source lakehouse table.
- `unified` = cleaned/conformed lakehouse table.
- `product` = curated product or warehouse output.
- `metadata` = governance evidence lakehouse. It is normally configured in `00_env_config` and is not usually selected as a business source table.

For the default one-source pipeline, keep `key` and `layer` as shown unless you know your pipeline reads from a different configured lakehouse layer. To support multiple source tables, add another dictionary to `SOURCE_TABLES` with a unique `key`. Do not copy profiling, schema, stability, DQ, or catalogue-evidence code.

Advanced override support: add `dataset_name`, `stage`, `watermark_value`, `dq_preset`, or `kind` inside a specific source table config only when that table needs to differ from the guardrail defaults.

In [ ]:
SOURCE_TABLES = [
    {
        "key": "source_01",
        "layer": "source",
        "table_name": "CHANGE_ME_source_table",
        "watermark_column": "CHANGE_ME_business_date",
        "expected_schema": {
            "customer_id": "bigint",
            "business_date": "date",
        },
    }
]

# To add a second source table, add another dictionary to SOURCE_TABLES:
# {
#     "key": "source_02",
#     "layer": "source",
#     "table_name": "CHANGE_ME_second_source_table",
#     "watermark_column": "CHANGE_ME_business_date",
#     "expected_schema": {"id": "bigint"},
# }
# Optional advanced per-table overrides, only when needed:
# "dataset_name": "CHANGE_ME_governance_dataset",
# "stage": "source",
# "watermark_value": "2026-01-31",
# "dq_preset": "approved_rules",
# "kind": "lakehouse",


## 5. Source guardrail defaults

These are the default guardrails applied to every source table. Most users should leave them as shown. Override a value inside a `SOURCE_TABLES` entry only when one source table needs a different schema rule, stability rule, DQ rule, profile distribution, or excluded column.

In [ ]:
DEFAULT_SOURCE_GUARDRAILS = {
    # Schema preset options:
    #   "allow_new_columns" = allow additive columns, block incompatible schema drift
    #   "strict" = require the schema to match exactly
    #   "monitor_only" = report schema differences without blocking
    "schema_preset": "allow_new_columns",

    # Data behavior options:
    #   "changing" = source data can change between runs
    #   "fixed" = source data is expected to remain stable
    "data_behavior": "changing",

    # Stability check options:
    #   "watermark_slice_hash" = hash one business-date/extract-date slice
    #   "full_profile_hash" = hash the full profile
    #   "skip" = skip stability enforcement for this table
    "stability_check_type": "watermark_slice_hash",

    # DQ preset options:
    #   "approved_rules" = enforce approved DQ rules from governance metadata
    #   "skip" = skip DQ enforcement for this table
    "dq_preset": "approved_rules",

    # Optional profile/stability guardrails.
    "distribution_columns": [],
    "exclude_columns": None,
}


## 6. Load source tables

This section reads each table listed in `SOURCE_TABLES` and adds the default source guardrails.
Most users do not need to edit this section. To use a file or warehouse source, comment out the default Lakehouse table read and uncomment the matching read option below.

In [ ]:
_SOURCE_TABLES_USER_CONFIG = SOURCE_TABLES
SOURCE_TABLES = []

for source_config in _SOURCE_TABLES_USER_CONFIG:
    dataset_name = source_config.get("dataset_name", source_config["table_name"])
    stage = source_config.get("stage", source_config["layer"])
    watermark_value = source_config.get("watermark_value", None)
    enriched_source = {
        **DEFAULT_SOURCE_GUARDRAILS,
        **source_config,
        "dataset_name": dataset_name,
        "stage": stage,
        "watermark_value": watermark_value,
    }

    # Default: read a Delta table from a configured Lakehouse Tables area.
    enriched_source["df"] = read_lakehouse_table(
        CONFIG,
        ENV_NAME,
        enriched_source["layer"],
        enriched_source["table_name"],
        spark_session=spark,
    )

    # CSV file alternative: read from a configured Lakehouse Files path.
    # enriched_source["df"] = read_lakehouse_csv(
    #     CONFIG,
    #     ENV_NAME,
    #     enriched_source["layer"],
    #     "CHANGE_ME/path/to/file.csv",
    #     spark_session=spark,
    #     header=True,
    # )

    # Parquet file alternative: read from a configured Lakehouse Files path.
    # enriched_source["df"] = read_lakehouse_parquet(
    #     CONFIG,
    #     ENV_NAME,
    #     enriched_source["layer"],
    #     "CHANGE_ME/path/to/file.parquet",
    #     verbose=True,
    #     spark_session=spark,
    # )

    # Excel file alternative: read from a configured Lakehouse Files path.
    # enriched_source["df"] = read_lakehouse_excel(
    #     CONFIG,
    #     ENV_NAME,
    #     enriched_source["layer"],
    #     "CHANGE_ME/path/to/file.xlsx",
    #     sheet_name=0,
    #     spark_session=spark,
    # )

    # Warehouse table alternative: read from a configured Fabric Warehouse target.
    # enriched_source["df"] = read_warehouse_table(
    #     CONFIG,
    #     ENV_NAME,
    #     "CHANGE_ME_warehouse_target",
    #     "dbo",
    #     enriched_source["table_name"],
    #     spark_session=spark,
    # )

    # Custom Spark table alternative: read a table available to the Spark session.
    # enriched_source["df"] = spark.read.table("CHANGE_ME_database.CHANGE_ME_table")

    SOURCE_TABLES.append(enriched_source)

SOURCE_CONFIG_BY_KEY = {source_config["key"]: source_config for source_config in SOURCE_TABLES}

# Convenience alias keeps the one-source starter transformation easy to read.
df_source_01 = SOURCE_CONFIG_BY_KEY["source_01"]["df"]


## 7. Optional: inspect a source schema

Run this cell while authoring if you want Spark to show the actual source schema before you finish `expected_schema` in the USER EDIT SECTION.

In [ ]:
# Optional authoring check: inspect the Spark schema before writing expected_schema.
df_source_01.printSchema()


## 8. Guardrail orchestration is handled by FabricOps

FabricOps runs profiling, schema validation, stability checks, DQ checks, catalogue evidence, and guardrail stopping through imported package helpers. Most users should not need to customize this orchestration code in the notebook.

## 9. Run source guardrails before transformation

Source profiling, schema validation, stability checks, DQ checks, and catalogue evidence run for every config in `SOURCE_TABLES`. Results remain separated by guardrail type and table key. Transformation starts only after `stop_if_any_guardrail_failed(source_guardrail_results)` passes.

In [ ]:
source_guardrail_results = run_table_guardrails(
    SOURCE_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(guardrail_summary(source_guardrail_results))
stop_if_any_guardrail_failed(source_guardrail_results)

# Runtime summary and lineage cells reuse these package-generated evidence objects.
source_schema_results = source_guardrail_results["schema_results"]
source_stability_results = source_guardrail_results["stability_results"]
source_dq_results = source_guardrail_results["dq_results"]
source_catalogue_status = source_guardrail_results["catalogue_status"]
source_evidence_definitions = source_guardrail_results["evidence_definitions"]

## 10. Transform to target DataFrames

This is the only section where most users write business transformation logic. Create one target DataFrame for each target table you plan to publish. FabricOps guardrails and audit columns are handled in later sections.

In [ ]:
# DIY your transformations here.
# Replace this passthrough with your business logic.
df_target_01 = df_source_01

# Example:
# df_target_01 = (
#     df_source_01
#     .select("customer_id", "business_date", "amount")
#     .where(F.col("amount").isNotNull())
# )

# Add more transformations or joins here. For many sources, use the config
# variables above or find a source by key from SOURCE_TABLES. For many targets,
# create df_target_02, df_target_03, and reference each DataFrame in TARGET_TABLES below.

## TARGET USER EDIT SECTION — target table configuration

Most users only edit this target section after creating target DataFrames in the transform section. Update each target `key`, `df`, `layer`, `table_name`, `write_mode`, `watermark_column`, and `expected_schema`. FabricOps derives the governance `dataset_name` from `table_name`, uses `layer` as the default governance `stage` and target write layer, uses `table_name` as the default target write name, uses `lakehouse` as the default target kind, and uses `None` as the default `watermark_value`.

For the default one-target pipeline, keep `df` as the DataFrame created in the transform section. To support multiple target tables, add another dictionary to `TARGET_TABLES` with a unique `key` and a DataFrame created in the transform section. Do not copy profiling, schema, stability, DQ, catalogue-evidence, or write orchestration code.

Advanced override support: add `dataset_name`, `stage`, `target_layer`, `target_name`, `target_kind`, `watermark_value`, `dq_preset`, or `kind` inside a specific target table config only when that table needs to differ from the guardrail or write defaults.

In [ ]:
TARGET_TABLES = [
    {
        "key": "target_01",
        "df": df_target_01,
        "layer": "unified",
        "table_name": "CHANGE_ME_target_table",
        "write_mode": "overwrite",
        "watermark_column": "CHANGE_ME_business_date",
        "expected_schema": {
            "customer_id": "bigint",
            "event_ts": "string",
            "status": "string",
            "amount": "double",
            "email": "string",
            "country_code": "string",
            "amount_band": "string",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
    }
]

# To add a second target table, create df_target_02 in the transform section,
# then add another dictionary to TARGET_TABLES:
# {
#     "key": "target_02",
#     "df": df_target_02,
#     "layer": "product",
#     "table_name": "CHANGE_ME_second_target_table",
#     "write_mode": "overwrite",
#     "watermark_column": "CHANGE_ME_business_date",
#     "expected_schema": {"id": "bigint"},
# }
# Optional advanced per-table overrides, only when needed:
# "dataset_name": "CHANGE_ME_governance_dataset",
# "stage": "product",
# "target_layer": "product",
# "target_name": "CHANGE_ME_written_table_name",
# "target_kind": "warehouse",
# "watermark_value": "2026-01-31",
# "dq_preset": "approved_rules",
# "kind": "warehouse",
# "partition_by": ["business_date"],
# "repartition_by": ["customer_id"],
# "overwrite_schema": True,


## Target guardrail and write defaults

These are the default guardrails and write options applied to every target table. Most users should leave them as shown. Override a value inside a `TARGET_TABLES` entry only when one target table needs a different schema rule, stability rule, DQ rule, profile distribution, excluded column, or write option.

In [ ]:
DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS = {
    # Schema preset options:
    #   "strict" = require the schema to match exactly
    #   "allow_new_columns" = allow additive columns, block incompatible schema drift
    #   "monitor_only" = report schema differences without blocking
    "schema_preset": "strict",

    # Data behavior options:
    #   "changing" = target data can change between runs
    #   "fixed" = target data is expected to remain stable
    "data_behavior": "changing",

    # Stability check options:
    #   "watermark_slice_hash" = hash one business-date/extract-date slice
    #   "full_profile_hash" = hash the full profile
    #   "skip" = skip stability enforcement for this table
    "stability_check_type": "watermark_slice_hash",

    # DQ preset options:
    #   "approved_rules" = enforce approved DQ rules from governance metadata
    #   "skip" = skip DQ enforcement for this table
    "dq_preset": "approved_rules",

    # Optional profile/stability guardrails.
    "distribution_columns": ["status", "amount", "amount_band", "country_code"],
    "exclude_columns": None,

    # Lakehouse write mode options: "overwrite", "append", "errorifexists", "ignore".
    # Warehouse writes use Spark connector modes such as "overwrite" or "append".
    "write_mode": "overwrite",

    # Optional Lakehouse write options.
    "partition_by": None,
    "repartition_by": None,
    "overwrite_schema": True,

    # Target kind options:
    #   "lakehouse" = write a Lakehouse Delta table
    #   "warehouse" = write a Fabric Warehouse table
    "kind": "lakehouse",
}


## TARGET FRAMEWORK PREPARATION — enrich target configs

Do not edit this section for normal target tables. It adds FabricOps runtime audit columns to each configured target DataFrame, applies `DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS`, derives `dataset_name`, `stage`, `target_layer`, `target_name`, `target_kind`, and `watermark_value` when they are not provided, and keeps downstream target profiling, schema validation, stability checks, DQ checks, catalogue evidence, and writes driven by `TARGET_TABLES`.

The `TARGET_01_*` variables created here are framework convenience aliases for starter notebook readability and lineage text. They are not user-edit fields.

In [ ]:
AUDIT_CREATED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
_TARGET_TABLES_USER_CONFIG = TARGET_TABLES
TARGET_TABLES = []

for target_config in _TARGET_TABLES_USER_CONFIG:
    target_df = (
        target_config["df"]
        .withColumn("_fabricops_run_id", F.lit(RUN_ID))
        .withColumn("_fabricops_pipeline_name", F.lit(PIPELINE_NAME))
        .withColumn("_fabricops_created_at", F.lit(AUDIT_CREATED_AT))
    )
    dataset_name = target_config.get("dataset_name", target_config["table_name"])
    stage = target_config.get("stage", target_config["layer"])
    target_layer = target_config.get("target_layer", target_config["layer"])
    target_name = target_config.get("target_name", target_config["table_name"])
    target_kind = target_config.get("target_kind", target_config.get("kind", "lakehouse"))
    watermark_value = target_config.get("watermark_value", None)
    enriched_target = {
        **DEFAULT_TARGET_GUARDRAILS_AND_WRITE_OPTIONS,
        **target_config,
        "df": target_df,
        "dataset_name": dataset_name,
        "stage": stage,
        "target_layer": target_layer,
        "target_name": target_name,
        "target_kind": target_kind,
        "watermark_value": watermark_value,
    }
    TARGET_TABLES.append(enriched_target)

TARGET_CONFIG_BY_KEY = {target_config["key"]: target_config for target_config in TARGET_TABLES}

# Framework convenience aliases keep starter lineage text and one-target writes easy to read.
TARGET_01_CONFIG = TARGET_CONFIG_BY_KEY["target_01"]
TARGET_01_KEY = TARGET_01_CONFIG["key"]
TARGET_01_TABLE_NAME = TARGET_01_CONFIG["table_name"]
TARGET_01_LAYER = TARGET_01_CONFIG["layer"]
TARGET_01_WRITE_MODE = TARGET_01_CONFIG["write_mode"]


## 13. Run target guardrails before writes

Target profiling, schema validation, stability checks, DQ checks, and catalogue evidence run for every config in `TARGET_TABLES`. Target writes do not happen unless `stop_if_any_guardrail_failed(target_guardrail_results)` passes.

Warning severity writes full data; error severity stops before write. No row filtering in v1.

In [ ]:
target_dq_results = {}
target_dfs = {target_config["key"]: target_config["df"] for target_config in TARGET_TABLES}
target_guardrail_results = run_table_guardrails(
    TARGET_TABLES,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    spark_session=spark,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)

display(guardrail_summary(target_guardrail_results))
target_profiles = target_guardrail_results["profiles"]
target_schema_results = target_guardrail_results["schema_results"]
target_stability_results = target_guardrail_results["stability_results"]
target_dq_results = target_guardrail_results["dq_results"]
for target_name in target_dq_results:
    print(target_dq_results[target_name])
    stop_if_failed(target_dq_results[target_name])
    if "dataframe" in target_dq_results[target_name]:
        target_dfs[target_name] = target_dq_results[target_name]["dataframe"]
target_catalogue_status = target_guardrail_results["catalogue_status"]
target_evidence_definitions = target_guardrail_results["evidence_definitions"]

stop_if_any_guardrail_failed(target_guardrail_results)


## 14. Write target tables

Only after all configured target guardrails pass, FabricOps loops through `TARGET_TABLES` and writes the target DataFrames. A blocking schema, stability, or DQ failure prevents every target write.

In [ ]:
target_write_status = {}
for target_config in TARGET_TABLES:
    target_key = target_config["key"]
    target_df = target_config["df"]
    target_kind = target_config.get("target_kind", "lakehouse")
    target_layer = target_config.get("target_layer", "unified")
    target_table = target_config.get("target_name", target_key)
    target_mode = target_config.get("write_mode", "overwrite")

    if target_kind == "lakehouse":
        write_lakehouse_table(
            target_df,
            CONFIG,
            ENV_NAME,
            target_layer,
            target_table,
            mode=target_mode,
            partition_by=target_config.get("partition_by"),
            repartition_by=target_config.get("repartition_by"),
            overwrite_schema=target_config.get("overwrite_schema", target_mode == "overwrite"),
        )
    elif target_kind == "warehouse":
        write_warehouse_table(
            target_df,
            CONFIG,
            ENV_NAME,
            target_layer,
            target_config.get("schema", "dbo"),
            target_table,
            mode=target_mode,
        )
    else:
        raise ValueError(f"Unsupported target kind for {target_key}: {target_kind}")
    target_write_status[target_key] = "written"


## 15. Capture many-to-many lineage

Define source-to-target relationships at the business level. FabricOps builds and writes metadata rows tied to the selected notebook registration.

In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": ["source_01"],
        "targets": [TARGET_01_KEY],
        "operation": f"derive amount band and publish {TARGET_01_TABLE_NAME}",
        "description": f"{SOURCE_CONFIG_BY_KEY['source_01']['table_name']} rows are transformed into {TARGET_01_TABLE_NAME}.",
    },
]

lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=TARGET_01_CONFIG["dataset_name"],
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 16. Write runtime summary

Runtime evidence is stored in `METADATA_PIPELINE_RUNS` and displayed for operational support.

In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    status="completed",
    source_definitions=source_evidence_definitions,
    target_definitions=target_evidence_definitions,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_stability_results=source_stability_results,
    target_stability_results=target_stability_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="Pipeline completed and metadata evidence was written.",
)

display(run_summary)
